# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Method:** this cell makes sure DuckDB is version 1.1 or newer (its Hugging Face login reads the token from the environment) and installs `shap`, which explains the models' scores page by page.


In [ ]:
%pip -q install "duckdb>=1.1" shap

## Setup

**Method, step by step:**

1. **Finds the project folder** and **reads the Hugging Face token**, as in the data contract: environment variable or `.env` file, then a Colab Secret named `HF_TOKEN`, then a typed prompt as the last resort. DuckDB reads the token from the environment, so it never appears in a query or an output.
2. **Loads the tools.** `duckdb` queries the warehouse, `pandas` and `numpy` handle the results, `scipy` supplies the negative binomial luck test behind the target and rule B, `scikit-learn` supplies the four models, the fold splitters, and the metrics, `shap` explains individual scores, and `matplotlib` draws the one chart.
3. **Defines the months and the cache.** The same three development pairs as the data contract; June 2026 is never read. Heavy queries go through `cached()`, which saves each result once under `work/outputs/w03_cache/` and reuses it on later runs.
4. **Defines `show()` and `show_table()`**, which display every result as a titled table.


In [ ]:
import getpass
import os
import sys
import time

import duckdb
import numpy as np
import pandas as pd
import shap
import sklearn
from IPython.display import Markdown, display
from scipy.stats import nbinom
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

## Data

The models are trained and tested on the same three development pairs as the data contract, with the same eligibility rules, the same spread factor method, and the same target. June 2026 stays sealed.


**Method:** this cell loads the page-month table for February to May 2026 with the same query and cache as the earlier notebooks (one row per page per month, position pieces only from days with a valid position). It also reads each client's search-history start date and, from the content table, the content type, the keyword-research fields used as features, and the latest optimization date used by the eligibility rule.


In [ ]:
# The same page-month query and cache as the earlier notebooks.
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)
assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
content = con.sql(f"SELECT content_hash_id, content_type, main_intent, search_volume, competition, cpc, keyword_token_count, last_optimized_date FROM {DIM_CONTENT}").df()
show_table("Inputs loaded", pd.DataFrame({
    "Rows": [f"{len(page_months):,}", f"{len(clients):,}", f"{len(content):,}"],
    "What it is": ["Page-months, February to May 2026", "Clients with their search-history start date",
                   "Pages with content type, keyword research fields, and latest optimization date"],
}, index=pd.Index(["page_months", "clients", "content"], name="Table")))

**Method:** this cell applies the data contract's rules to each development pair (impressions in both months, client history covering all of month M, a valid position in both months, not optimized during the pair). For each month it computes the impression-weighted position, its daily spread, expected clicks from the fine position bins and from the five tiers (both client-scaled), and whether the page fell short by the locked rule: clicks at most half of expected, with negative binomial p below 0.05. The spread factor α is estimated exactly as the data contract does (one estimate per pair from pages whose position barely moved, the median rounded to three decimals), so every notebook shares one target. The month M+1 version of the rule is the target.


In [ ]:
# Rules and bins exactly as in the data contract; the spread factor is estimated below with its method.
BIN_EDGES = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 20.5, 50.5]
BIN_NAMES = [str(position) for position in range(1, 11)] + ["11-20", "21-50", "50+"]


def position_bin(position: pd.Series) -> pd.Series:
    """Return each page's position bin: exact rank up to 10, then 11-20, 21-50, and 50+."""
    return pd.Series(np.array(BIN_NAMES)[np.searchsorted(BIN_EDGES, position, side="right")], index=position.index)


def position_tier(position: pd.Series) -> pd.Series:
    """Return the five tiers used by the earlier score: below 3, below 10, below 20, below 50, and 50 or beyond."""
    tiers = np.select([position < 3, position < 10, position < 20, position < 50], ["top_3", "page_1", "striking", "page_3_5"], "deep")
    return pd.Series(tiers, index=position.index)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str, group: pd.Series) -> pd.Series:
    """Return expected clicks from each group's pooled click rate, scaled by the page's client factor (+1 on both sides)."""
    rate = frame[clicks].groupby(group).sum() / frame[impressions].groupby(group).sum()
    expected = frame[impressions] * group.map(rate)
    by_client = frame.assign(_expected=expected).groupby("client_hash_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_hash_id"].map(factor)


def covered_clients(month_m: str) -> pd.Series:
    """Return the clients whose search history starts on or before the first day of month M."""
    return clients.loc[clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"), "client_hash_id"]


def add_month_measures(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add position, its spread, click-through, and both expectations for one month."""
    impressions, clicks = rows[f"impressions_{suffix}"], rows[f"clicks_{suffix}"]
    rows[f"position_{suffix}"] = rows[f"sum_position_{suffix}"] / rows[f"position_impressions_{suffix}"]
    spread = rows[f"sum_position_sq_{suffix}"] / rows[f"position_impressions_{suffix}"] - rows[f"position_{suffix}"] ** 2
    rows[f"position_volatility_{suffix}"] = np.sqrt(spread.clip(lower=0))
    rows[f"ctr_{suffix}"] = clicks / impressions
    rows[f"expected_bins_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_bin(rows[f"position_{suffix}"]))
    rows[f"expected_tier_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_tier(rows[f"position_{suffix}"]))
    return rows


def add_luck_test(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add the negative binomial luck test and the locked shortfall rule for one month."""
    clicks, expected = rows[f"clicks_{suffix}"], rows[f"expected_bins_{suffix}"]
    rows[f"p_{suffix}"] = nbinom.cdf(clicks, 1 / ALPHA, 1 / (1 + ALPHA * expected))
    rows[f"shortfall_{suffix}"] = (rows[f"p_{suffix}"] < 0.05) & (clicks <= 0.5 * expected)
    return rows


def build_pair(month_m: str, month_next: str) -> pd.DataFrame:
    """Return one month pair's eligible rows with both months' measures and the locked target."""
    keys = ["client_hash_id", "content_hash_id"]
    first = page_months[page_months["month"] == month_m].set_index(keys)
    second = page_months[page_months["month"] == month_next].set_index(keys)
    rows = first.join(second, lsuffix="_m", rsuffix="_next", how="inner").reset_index()
    start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
    rows = rows[rows["client_hash_id"].isin(covered_clients(month_m))]
    rows = rows[(rows["position_impressions_m"] > 0) & (rows["position_impressions_next"] > 0)]
    rows = rows.merge(content, on="content_hash_id", how="left")
    rows = rows[~pd.to_datetime(rows["last_optimized_date"]).between(start, end)].copy()
    rows = add_month_measures(add_month_measures(rows, "m"), "next")
    rows["pair_end"] = end
    return rows.reset_index(drop=True)


def stable_pages(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Return pages whose position moved less than one place, and each one's expected share of clicks in month M+1.

    The expected share corrects for the overall change in click rate between the two months.
    """
    stable = frame[(frame["position_next"] - frame["position_m"]).abs() < 1]
    shift = (stable["clicks_next"].sum() / stable["impressions_next"].sum()) / (stable["clicks_m"].sum() / stable["impressions_m"].sum())
    share = shift * stable["impressions_next"] / (stable["impressions_m"] + shift * stable["impressions_next"])
    return stable, share



def spread_factor(frame: pd.DataFrame) -> float:
    """Return the method-of-moments estimate of the negative binomial spread factor alpha.

    For pages with a stable position, the gap between month-M+1 clicks and the clicks their month-M rate
    predicts has variance (Poisson part) + alpha x (squared clicks). Pooling the excess over the Poisson
    part across pages gives alpha.
    """
    stable, share = stable_pages(frame)
    scale = share / (1 - share)
    gap = stable["clicks_next"] - scale * stable["clicks_m"]
    excess = (gap ** 2 - stable["clicks_next"] - scale ** 2 * stable["clicks_m"]).sum()
    squared = ((stable["clicks_next"] ** 2 - stable["clicks_next"]) + scale ** 2 * (stable["clicks_m"] ** 2 - stable["clicks_m"])).sum()
    return float(excess / squared)



pairs = {f"{month_m} → {month_next}": build_pair(month_m, month_next) for month_m, month_next in PAIRS}
ALPHA = round(float(np.median([spread_factor(rows) for rows in pairs.values()])), 3)
for rows in pairs.values():
    add_luck_test(add_luck_test(rows, "m"), "next")
    rows["target"] = rows["shortfall_next"]
    rows["missed_next"] = (rows["expected_bins_next"] - rows["clicks_next"]).clip(lower=0).where(rows["target"], 0)
show_table(f"Development pairs with the locked target (α = {ALPHA:.3f})", pd.DataFrame({
    "Rows": [f"{len(rows):,}" for rows in pairs.values()],
    "Positives": [f"{rows['target'].sum():,}" for rows in pairs.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows in pairs.values()],
}, index=pd.Index(list(pairs), name="Pair")))

**Method:** this cell builds the feature vector: the leakage check's 23 numeric features (month M search data and keyword research done before the page was written, with keyword gaps filled by 0 and marked by a has-value flag) plus two signals from rule B, both from month M: the page's missed clicks against the position-bin expectation (log scale), and whether it fell short by the target's rule. An earlier run of this notebook showed the model judging shortfalls by ratio and passing over very large pages; these two features let it see the size of the gap, which is what recovering clicks depends on. The cell checks that every pair yields the same columns with no gaps and that no feature comes from month M+1 or from a FlyRank product field.


In [ ]:
# The leakage check's 23 features, plus rule B's two month-M signals.
CONTENT_TYPES = sorted(content["content_type"].dropna().unique())
INTENTS = sorted(content["main_intent"].dropna().unique())
PRODUCT_FIELDS = {"last_optimized_date", "optimization_eligible_date", "is_deleted", "is_published", "content_updated_date"}


def feature_vector(rows: pd.DataFrame) -> pd.DataFrame:
    """Return the model-ready feature matrix for one pair: month-M search features and keyword-research features only."""
    features = pd.DataFrame(index=rows.index)
    features["log_impressions_m"] = np.log1p(rows["impressions_m"])
    features["log_clicks_m"] = np.log1p(rows["clicks_m"])
    features["ctr_m"] = rows["clicks_m"] / rows["impressions_m"]
    features["position_m"] = rows["position_m"]
    features["position_volatility_m"] = rows["position_volatility_m"]
    features["days_with_impressions_m"] = rows["days_with_impressions_m"]
    features["log_expected_m"] = np.log1p(rows["expected_tier_m"])
    for field in ("search_volume", "competition", "cpc", "main_intent"):
        features[f"has_{field}"] = rows[field].notna().astype(int)
    features["log_search_volume"] = np.log1p(rows["search_volume"]).fillna(0)
    features["competition"] = rows["competition"].fillna(0)
    features["log_cpc"] = np.log1p(rows["cpc"]).fillna(0)
    features["keyword_token_count"] = rows["keyword_token_count"]
    for value in CONTENT_TYPES:
        features[f"content_type={value}"] = (rows["content_type"] == value).astype(int)
    for value in INTENTS:
        features[f"intent={value}"] = (rows["main_intent"] == value).astype(int)
    features["intent=unknown"] = rows["main_intent"].isna().astype(int)
    features["log_missed_m"] = np.log1p((rows["expected_bins_m"] - rows["clicks_m"]).clip(lower=0))
    features["short_m"] = rows["shortfall_m"].astype(int)
    return features


vectors = {name: feature_vector(rows) for name, rows in pairs.items()}
FEATURES = list(next(iter(vectors.values())).columns)
for name, features in vectors.items():
    assert list(features.columns) == FEATURES, f"{name}: columns differ from the first pair."
    assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in features.dtypes), f"{name}: a feature is not numeric."
    assert not features.isna().any().any(), f"{name}: a feature still has gaps."
assert not [column for column in FEATURES if column.endswith("_next")], "A month M+1 field reached the features."
assert not set(FEATURES) & PRODUCT_FIELDS, "A product field reached the features."

LABELS = {"log_impressions_m": "Impressions (log)", "log_clicks_m": "Clicks (log)", "ctr_m": "Click-through rate",
          "position_m": "Position", "position_volatility_m": "Position spread", "days_with_impressions_m": "Days with impressions",
          "log_expected_m": "Expected clicks (log)", "has_search_volume": "Has search volume", "has_competition": "Has competition",
          "has_cpc": "Has CPC", "has_main_intent": "Has search intent", "log_search_volume": "Search volume (log)",
          "competition": "Competition", "log_cpc": "CPC (log)", "keyword_token_count": "Keyword length (words)",
          "log_missed_m": "Missed clicks (log)", "short_m": "Short by the target's rule"}


def label(feature: str) -> str:
    """Return a plain-language name for a feature."""
    return LABELS.get(feature) or feature.replace("content_type=", "Content type: ").replace("intent=", "Search intent: ")
show_table("Feature vector, every pair", pd.DataFrame({
    "Rows": [f"{len(features):,}" for features in vectors.values()],
    "Features": [features.shape[1] for features in vectors.values()],
    "Gaps left": [int(features.isna().sum().sum()) for features in vectors.values()],
}, index=pd.Index(list(vectors), name="Pair")))

## 1. Method choice and why

The question is a ranking: _which pages should an editor look at first?_ Every model below outputs a probability that a page will be a real shortfall next month, and the pages are ranked by it, exactly as rule B ranks them by missed clicks. They are judged by Precision@50, as the rule was.

| Model                   | Why it is in the lineup                                                                                                                 | Settings, fixed before any result                                                                                                                                         |
| ----------------------- | --------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Logistic regression** | The readable reference: one weight per feature, no interactions                                                                         | Features standardized (they are already on a log scale where needed); default regularization, C = 1                                                                       |
| **Decision tree**       | Rules a person can read in one screen                                                                                                   | Depth 3 (at most 8 rules), at least 50 pages per leaf                                                                                                                     |
| **Random forest**       | Learns combinations such as "an unstable rank matters only for big pages" (the signal audit found exactly this)                         | 300 trees, at least 50 pages per leaf, each tree grown on a random quarter of the training rows, features per split at the default (square root of the count)             |
| **Gradient boosting**   | Usually the strongest model family on table data (Grinsztajn et al., 2022); the leakage check's probe reached Precision@50 0.86 with it | Default learning rate (0.1) and tree size; up to 300 trees, the number chosen on held-out _training clients_, with the loss averaged over five client-grouped inner folds |

**Why the settings are fixed rather than tuned.** Choosing settings by their score on the test pairs would quietly turn the test pairs into training data and overstate the result (Cawley and Talbot, 2010). Rule B was not tuned either, so untuned models are the fair comparison. Random forests are known to work well with standard settings (Probst, Boulesteix and Bischl, 2019); gradient boosting gains more from tuning, so its result here is, if anything, an underestimate. Each forest tree sees a random quarter of the training rows instead of all of them, which keeps the run to minutes on Colab; with 300 trees the averaging absorbs the extra variation of each tree.

**Two further choices.** No class weights and no resampling: the target is rare (4 to 7%), but only the _order_ of the pages matters for Precision@50, and reweighting would distort the probabilities without changing what a ranking needs. Every random step uses the seed 42.

**Method:** this cell defines the four models with their settings, the early-stopping procedure for gradient boosting, and rule B, and records the seed and library versions.


In [ ]:
SEED = 42
MODEL_ORDER = ["Logistic regression", "Decision tree", "Random forest", "Gradient boosting"]  # simplest first
BASELINE = "Rule B (baseline)"
MAX_TREES = 300


def make_model(name: str) -> object:
    """Return a fresh, unfitted model with its fixed settings."""
    return {
        "Logistic regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2_000)),
        "Decision tree": lambda: DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED),
        "Random forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=50, max_samples=0.25, n_jobs=-1, random_state=SEED),
    }[name]()


def fit_boosting(features: pd.DataFrame, target: pd.Series, groups: pd.Series) -> tuple[HistGradientBoostingClassifier, int]:
    """Fit gradient boosting with the number of trees chosen on held-out training clients, then refit on all training clients.

    scikit-learn's built-in early stopping holds out random rows, which would put the same client on both
    sides; holding out whole clients keeps the stopping decision honest. The loss curve is averaged over
    five client-grouped inner folds, because one held-out group of a few clients gives an unstable choice.
    """
    curves = []
    for inner, held in GroupKFold(n_splits=5).split(features, target, groups):
        trial = HistGradientBoostingClassifier(max_iter=MAX_TREES, early_stopping=False, random_state=SEED)
        trial.fit(features.iloc[inner], target.iloc[inner])
        curves.append([log_loss(target.iloc[held], stage[:, 1], labels=[0, 1]) for stage in trial.staged_predict_proba(features.iloc[held])])
    trees = int(np.argmin(np.mean(curves, axis=0))) + 1
    final = HistGradientBoostingClassifier(max_iter=trees, early_stopping=False, random_state=SEED).fit(features, target)
    return final, trees


def rule_b(rows: pd.DataFrame) -> pd.Series:
    """The frozen baseline: missed clicks against the position-bin expectation, kept where the page fell short by the target's rule in month M."""
    missed = (rows["expected_bins_m"] - rows["clicks_m"]).clip(lower=0)
    return missed.where(rows["shortfall_m"], 0)


show_table("The four models and their fixed settings", pd.DataFrame({
    "Settings": ["standardized features, C = 1", "depth 3, at least 50 pages per leaf",
                 "300 trees, at least 50 pages per leaf, a quarter of the rows per tree",
                 f"learning rate 0.1, up to {MAX_TREES} trees chosen on held-out training clients (5 inner folds)"],
}, index=pd.Index(MODEL_ORDER, name="Model")))
show("Reproducibility", {"seed": SEED, "scikit-learn": sklearn.__version__, "shap": shap.__version__,
                         "pandas": pd.__version__, "numpy": np.__version__})

## 2. Split design

**The question the split must answer:** _would this model, built at the end of one month from what was known then, rank next month's real shortfalls on a site it has never seen?_ That needs two protections at once.

1. **Walk-forward in time.** Each test pair is scored by models trained only on earlier pairs: Feb→Mar trains the models that score Mar→Apr, and Feb→Mar plus Mar→Apr train the models that score Apr→May. The training targets end on the last day of the test pair's month M, so everything a model learns from was known when the queue would be built. The first pair is never a test pair, because nothing comes before it.
2. **Grouped by client.** The test pair's clients are split into five folds. For each fold, the models train on the earlier pairs _without_ that fold's clients, then score that fold's pages. Every test page is therefore scored by a model that never saw its client, in any month.

Rule B needs no training, so it scores every test page directly. All five scorers are judged on exactly the same pages. The two test pairs are reported separately as well as averaged, because the first one trains on less data.

**Method:** this cell builds the client folds for each test pair and checks both protections: no fold's clients appear in its training rows, and no training target falls after the test pair's month M.


In [ ]:
# Walk-forward: each test pair is trained on every earlier pair.
names = list(pairs)
TESTS = {names[i]: names[:i] for i in range(1, len(names))}
folds = {test: list(GroupKFold(n_splits=5).split(pairs[test], groups=pairs[test]["client_hash_id"])) for test in TESTS}

plan = []
for test, trained_on in TESTS.items():
    test_rows = pairs[test]
    train_rows = pd.concat([pairs[name] for name in trained_on], ignore_index=True)
    month_m_end = pd.Timestamp(f"{test.split(' → ')[0]}-01") + pd.offsets.MonthEnd(0)
    assert (train_rows["pair_end"] <= month_m_end).all(), f"{test}: a training target falls after month M."
    fold_clients = [set(test_rows.loc[test_index, "client_hash_id"]) for _, test_index in folds[test]]
    overlap = sum(train_rows.loc[~train_rows["client_hash_id"].isin(held), "client_hash_id"].isin(held).sum() for held in fold_clients)
    assert overlap == 0, f"{test}: a held-out client reached its training rows."
    plan.append({"Test pair": test, "Trained on": " + ".join(trained_on), "Training rows (all clients)": f"{len(train_rows):,}",
                 "Test rows": f"{len(test_rows):,}", "Test clients": test_rows["client_hash_id"].nunique(),
                 "Clients per fold": f"{min(map(len, fold_clients))} to {max(map(len, fold_clients))}",
                 "Test pages per fold": f"{min(len(index) for _, index in folds[test]):,} to {max(len(index) for _, index in folds[test]):,}",
                 "Held-out clients in training": overlap})
show_table("Walk-forward, client-grouped split", pd.DataFrame(plan).set_index("Test pair"))

## 3. Train + compare vs my baseline

**The comparison rule, fixed before the results** (the same yardstick that froze rule B): a model beats rule B if its mean Precision@50 over the two test pairs is more than 0.02 higher; within 0.02 the two count as tied, and the higher mean Capture@50 decides. Of the models that beat rule B, the one carried forward is the best by mean Precision@50, except that a simpler model wins whenever it is within 0.02 of the best (logistic regression, then the tree, the forest, and boosting). If no model beats rule B, rule B stays.

**Metrics:** Precision@50 (primary) and Precision@20; Capture@50, the share of next month's missed clicks on real shortfalls that the top 50 hold, because the target favours large pages; Precision@50 within the fixed size band of 1,000 to 9,999 impressions, so a model cannot win by picking bigger pages; and two whole-ranking diagnostics, ROC-AUC and average precision. The base rate stands beside them all. Precision is tie-aware, so the decision tree, which gives whole leaves the same score, is credited with the share of positives among the pages tied at the cut-off rather than with an arbitrary order.


### 3a. Train the four models and compare them with rule B

**Method:** this cell defines the tie-aware metrics and one shared procedure that trains a model on every client fold of both test pairs and scores the held-out pages. It then scores rule B on the same pages. The four cells after it each train one model with that procedure and show its first results.


In [ ]:
def top_k_total(score: pd.Series, values: pd.Series, k: int) -> float:
    """Return the expected total of `values` over the top k rows, counting rows tied at the cut-off by their average."""
    k = min(k, len(score))  # a subset smaller than k is judged on all its rows
    cut = score.sort_values(ascending=False).iloc[k - 1]
    above = values[score > cut]
    return float(above.sum() + (k - len(above)) * values[score == cut].mean())


def precision_at_k(score: pd.Series, target: pd.Series, k: int) -> float:
    """Return the tie-aware share of the top k rows whose target is true."""
    return top_k_total(score, target.astype(float), k) / k


def capture_at_k(score: pd.Series, value: pd.Series, k: int) -> float:
    """Return the tie-aware share of the total value held by the top k rows."""
    return top_k_total(score, value, k) / float(value.sum())


def metrics(test: str, scorer: str) -> dict[str, float]:
    """Return every comparison metric for one scorer on one test pair."""
    rows, score = pairs[test], scores[(test, scorer)]
    band = rows["impressions_m"].between(1_000, 9_999)
    return {"Precision@50": precision_at_k(score, rows["target"], 50),
            "Precision@20": precision_at_k(score, rows["target"], 20),
            "Capture@50": capture_at_k(score, rows["missed_next"], 50),
            "Precision@50, 1,000 to 9,999 impressions": precision_at_k(score[band], rows.loc[band, "target"], 50),
            "ROC-AUC": roc_auc_score(rows["target"], score),
            "Average precision": average_precision_score(rows["target"], score),
            "Base rate": rows["target"].mean()}


def show_first_results(scorer: str, note: dict[str, object] | None = None) -> None:
    """Display one scorer's headline metrics on each test pair."""
    table = pd.DataFrame({test: metrics(test, scorer) for test in TESTS}).T
    show_table(f"{scorer}: first results", pd.DataFrame({
        "Precision@50": table["Precision@50"].map("{:.2f}".format),
        "Precision@20": table["Precision@20"].map("{:.2f}".format),
        "Capture@50": table["Capture@50"].map("{:.1%}".format),
        "Base rate": table["Base rate"].map("{:.1%}".format),
    }).rename_axis("Test pair"))
    if note:
        show(f"{scorer}: training details", note)


def train_and_score(model_name: str) -> list[int]:
    """Train one model on every client fold of every test pair and score the held-out pages.

    Returns the number of trees chosen per fold for gradient boosting, and an empty list for the other models.
    """
    trees_chosen = []
    for test, trained_on in TESTS.items():
        test_rows, test_features = pairs[test], vectors[test]
        train_rows = pd.concat([pairs[name] for name in trained_on], ignore_index=True)
        train_features = pd.concat([vectors[name] for name in trained_on], ignore_index=True)
        train_target = train_rows["target"].astype(int)
        score, fitted[(test, model_name)] = pd.Series(np.nan, index=test_rows.index), []
        for _, test_index in folds[test]:
            keep = ~train_rows["client_hash_id"].isin(set(test_rows.loc[test_index, "client_hash_id"]))
            if model_name == "Gradient boosting":
                model, trees = fit_boosting(train_features[keep], train_target[keep], train_rows.loc[keep, "client_hash_id"])
                trees_chosen.append(trees)
            else:
                model = make_model(model_name).fit(train_features[keep], train_target[keep])
            score.iloc[test_index] = model.predict_proba(test_features.iloc[test_index])[:, 1]
            fitted[(test, model_name)].append((model, test_index))
        assert not score.isna().any(), f"{test}, {model_name}: a test page was left unscored."
        scores[(test, model_name)] = score
    return trees_chosen


scores, fitted = {}, {}
for test in TESTS:
    scores[(test, BASELINE)] = rule_b(pairs[test])
show_first_results(BASELINE)

**Method:** this cell trains logistic regression on every client fold and shows its first results.


In [ ]:
# Logistic regression: one weight per standardized feature.
started = time.time()
train_and_score("Logistic regression")
show_first_results("Logistic regression", {"training time": f"{time.time() - started:.0f} seconds"})

**Method:** this cell trains the depth-3 decision tree on every client fold and shows its first results.


In [ ]:
# Decision tree: at most 8 leaves; pages in the same leaf share one score.
started = time.time()
train_and_score("Decision tree")
show_first_results("Decision tree", {"training time": f"{time.time() - started:.0f} seconds"})

**Method:** this cell trains the random forest on every client fold and shows its first results. It is the slowest model: 300 trees per fold, ten folds.


In [ ]:
# Random forest: 300 trees per fold, each on a random quarter of the training rows.
started = time.time()
train_and_score("Random forest")
show_first_results("Random forest", {"training time": f"{time.time() - started:.0f} seconds"})

**Method:** this cell trains gradient boosting on every client fold, choosing the number of trees each time from the loss averaged over five client-grouped inner folds of the training data, and shows its first results with the number of trees chosen.


In [ ]:
# Gradient boosting: trees added until held-out training clients stop improving.
started = time.time()
trees_chosen = train_and_score("Gradient boosting")
show_first_results("Gradient boosting", {"training time": f"{time.time() - started:.0f} seconds",
                                         "trees chosen per fold": ", ".join(map(str, trees_chosen)),
                                         "out of a maximum of": MAX_TREES})

**Method:** this cell gathers every metric for rule B and the four models into the one comparison table, computed in this run on exactly the same test pages.


In [ ]:
SCORERS = [BASELINE, *MODEL_ORDER]
results = pd.DataFrame([{"Test pair": test, "Scorer": scorer, **metrics(test, scorer)} for test in TESTS for scorer in SCORERS])
show_table("Rule B and the four models on the same test pages", results.assign(
    **{column: results[column].map("{:.2f}".format) for column in ["Precision@50", "Precision@20", "Precision@50, 1,000 to 9,999 impressions"]},
    **{column: results[column].map("{:.3f}".format) for column in ["ROC-AUC", "Average precision"]},
    **{"Capture@50": results["Capture@50"].map("{:.1%}".format), "Base rate": results["Base rate"].map("{:.1%}".format)},
).set_index(["Test pair", "Scorer"]))

**Method:** this cell averages each scorer over the two test pairs and applies the comparison rule fixed above: which models beat rule B, and which scorer is carried forward.


In [ ]:
means = results.groupby("Scorer")[["Precision@50", "Precision@20", "Capture@50", "Precision@50, 1,000 to 9,999 impressions",
                                    "Average precision"]].mean().loc[SCORERS]


def beats(challenger: pd.Series, holder: pd.Series) -> bool:
    """Apply the comparison rule: more than 0.02 higher mean Precision@50, or within 0.02 and a higher mean Capture@50."""
    gap = round(challenger["Precision@50"] - holder["Precision@50"], 6)  # rounded so "within 0.02" is not decided by float noise
    return gap > 0.02 or (abs(gap) <= 0.02 and challenger["Capture@50"] > holder["Capture@50"])


winners = [name for name in MODEL_ORDER if beats(means.loc[name], means.loc[BASELINE])]
if winners:
    best = round(means.loc[winners, "Precision@50"].max(), 6)
    CARRIED = next(name for name in MODEL_ORDER if name in winners and round(means.loc[name, "Precision@50"], 6) >= round(best - 0.02, 6))
else:
    CARRIED = BASELINE
# The model whose errors and explanations are read below: the carried model, or the strongest model if rule B stays.
EXPLAINED = CARRIED if CARRIED != BASELINE else means.loc[MODEL_ORDER, "Precision@50"].idxmax()

show_table("Means over the two test pairs, and the verdict", pd.DataFrame({
    "Mean Precision@50": means["Precision@50"].map("{:.3f}".format),
    "Gap to rule B": (means["Precision@50"] - means.loc[BASELINE, "Precision@50"]).map("{:+.3f}".format),
    "Mean Precision@20": means["Precision@20"].map("{:.3f}".format),
    "Mean Capture@50": means["Capture@50"].map("{:.2%}".format),
    "Mean Precision@50, 1,000 to 9,999": means["Precision@50, 1,000 to 9,999 impressions"].map("{:.3f}".format),
    "Mean average precision": means["Average precision"].map("{:.3f}".format),
    "Beats rule B": ["" if name == BASELINE else ("yes" if name in winners else "no") for name in SCORERS],
    "Carried forward": ["yes" if name == CARRIED else "" for name in SCORERS],
}).rename_axis("Scorer"))
show("Verdict", {"carried forward": CARRIED, "model read in section 4": EXPLAINED})

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.